# Tissue Routing Analysis for Hierarchical THERAPI

This notebook analyzes the tissue routing behavior of Hierarchical THERAPI:
1. Visualize tissue routing weights
2. Analyze routing accuracy
3. Examine attention patterns within tissues
4. Compare to biological expectations

In [ ]:
import sys
import os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from utils.data_loader import TransactDataLoader
from utils.tissue_mapping import TissueMapper
from models.hierarchical_therapi import HierarchicalTHERAPI

%matplotlib inline
plt.style.use('seaborn-v0_8-paper')
sns.set_palette('husl')

## 1. Load Model and Data

In [ ]:
# Load trained model
model_path = '../ckpts/HierarchicalTHERAPI_aligner_GDSC_TCGA.pt'
checkpoint = torch.load(model_path, map_location='cpu')

# Initialize tissue mapper
tissue_mapper = checkpoint.get('tissue_mapper', TissueMapper())
common_genes = checkpoint.get('common_genes', None)
config = checkpoint.get('config', {})

print(f"Model trained for {checkpoint['epoch']} epochs")
print(f"Number of tissue groups: {tissue_mapper.n_tissue_groups}")
print(f"Common genes: {len(common_genes) if common_genes else 'N/A'}")

In [ ]:
# Load data
data_loader = TransactDataLoader('../data/')
tcga_data = data_loader.load_tcga_data()
gdsc_data = data_loader.load_gdsc_data()

# Prepare data
if common_genes:
    tcga_expr = tcga_data['expression'][common_genes]
    gdsc_expr = gdsc_data['expression'][common_genes]
else:
    tcga_expr = tcga_data['expression']
    gdsc_expr = gdsc_data['expression']

print(f"TCGA samples: {tcga_expr.shape[0]}")
print(f"GDSC cell lines: {gdsc_expr.shape[0]}")

In [ ]:
# Initialize and load model
tissue_cell_mask, _ = tissue_mapper.create_cell_line_tissue_matrix(gdsc_data['tissue_mapping'])
tissue_cell_mask_tensor = torch.tensor(tissue_cell_mask, dtype=torch.float32)

model = HierarchicalTHERAPI(
    n_genes=len(common_genes) if common_genes else gdsc_expr.shape[1],
    n_tissues=tissue_mapper.n_tissue_groups,
    n_latent=config.get('latent_dim', 128),
    tissue_cell_mask=tissue_cell_mask_tensor
)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("Model loaded successfully!")

## 2. Analyze Tissue Routing Weights

In [ ]:
# Compute routing weights for all TCGA samples
tcga_tensor = torch.tensor(tcga_expr.values[:100], dtype=torch.float32)  # First 100 for speed
gdsc_tensor = torch.tensor(gdsc_expr.values, dtype=torch.float32)

with torch.no_grad():
    output = model.hierarchical_attention_forward(
        tcga_tensor,
        gdsc_tensor,
        return_attention=True
    )

tissue_weights = output['tissue_weights'].numpy()
print(f"Tissue weights shape: {tissue_weights.shape}")

In [ ]:
# Visualize tissue routing distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Heatmap of routing weights
tissue_names = [tissue_mapper.get_tissue_name_from_idx(i) for i in range(tissue_mapper.n_tissue_groups)]
sns.heatmap(tissue_weights[:50].T, ax=axes[0, 0], cmap='viridis',
           yticklabels=tissue_names, xticklabels=False,
           cbar_kws={'label': 'Routing Weight'})
axes[0, 0].set_title('Tissue Routing Weights (First 50 Samples)')
axes[0, 0].set_ylabel('Tissue Type')

# 2. Average routing weight per tissue
avg_weights = tissue_weights.mean(axis=0)
axes[0, 1].bar(range(len(avg_weights)), avg_weights)
axes[0, 1].set_xticks(range(len(tissue_names)))
axes[0, 1].set_xticklabels(tissue_names, rotation=45, ha='right')
axes[0, 1].set_title('Average Routing Weight per Tissue')
axes[0, 1].set_ylabel('Average Weight')

# 3. Distribution of max routing weights
max_weights = tissue_weights.max(axis=1)
axes[1, 0].hist(max_weights, bins=30, edgecolor='black')
axes[1, 0].set_title('Distribution of Maximum Routing Weights')
axes[1, 0].set_xlabel('Max Weight')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].axvline(max_weights.mean(), color='red', linestyle='--', label=f'Mean: {max_weights.mean():.3f}')
axes[1, 0].legend()

# 4. Entropy of routing weights
epsilon = 1e-8
entropy = -np.sum(tissue_weights * np.log(tissue_weights + epsilon), axis=1)
axes[1, 1].hist(entropy, bins=30, edgecolor='black')
axes[1, 1].set_title('Entropy of Routing Weights')
axes[1, 1].set_xlabel('Entropy')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].axvline(entropy.mean(), color='red', linestyle='--', label=f'Mean: {entropy.mean():.3f}')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('../output/tissue_routing_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Routing Accuracy Analysis

In [ ]:
# Get true tissue labels for TCGA samples
if 'tissue_info' in tcga_data and tcga_data['tissue_info'] is not None:
    tissue_info = tcga_data['tissue_info']
    
    # Match samples
    matched_samples = []
    true_tissues = []
    pred_tissues = []
    
    for i, sample_id in enumerate(tcga_expr.index[:100]):
        if sample_id in tissue_info.iloc[:, 0].values:
            true_tissue_name = tissue_info[tissue_info.iloc[:, 0] == sample_id].iloc[0, 1]
            true_tissue_idx = tissue_mapper.get_tissue_idx(true_tissue_name)
            pred_tissue_idx = tissue_weights[i].argmax()
            
            matched_samples.append(sample_id)
            true_tissues.append(true_tissue_idx)
            pred_tissues.append(pred_tissue_idx)
    
    # Compute accuracy
    accuracy = np.mean(np.array(true_tissues) == np.array(pred_tissues))
    print(f"Tissue Routing Accuracy: {accuracy:.2%}")
    print(f"Number of matched samples: {len(matched_samples)}")
else:
    print("No tissue info available for accuracy calculation")

In [ ]:
# Confusion matrix
if len(true_tissues) > 0:
    from sklearn.metrics import confusion_matrix
    
    cm = confusion_matrix(true_tissues, pred_tissues)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=tissue_names, yticklabels=tissue_names)
    plt.title('Tissue Routing Confusion Matrix')
    plt.xlabel('Predicted Tissue')
    plt.ylabel('True Tissue')
    plt.tight_layout()
    plt.savefig('../output/tissue_routing_confusion.png', dpi=300, bbox_inches='tight')
    plt.show()

## 4. Attention Pattern Analysis

In [ ]:
# Analyze attention breakdown for sample patients
attention_breakdown = output.get('attention_breakdown', {})

if attention_breakdown:
    # Pick a sample
    sample_idx = 0
    
    print(f"Attention analysis for sample {sample_idx}:")
    print(f"Tissue weights: {tissue_weights[sample_idx]}")
    print(f"\nTop 3 tissues:")
    
    top_tissues = tissue_weights[sample_idx].argsort()[-3:][::-1]
    for rank, tissue_idx in enumerate(top_tissues, 1):
        tissue_name = tissue_mapper.get_tissue_name_from_idx(tissue_idx)
        weight = tissue_weights[sample_idx, tissue_idx]
        if tissue_idx in attention_breakdown:
            n_cells = attention_breakdown[tissue_idx]['n_cells']
            print(f"{rank}. {tissue_name}: weight={weight:.4f}, n_cells={n_cells}")
        else:
            print(f"{rank}. {tissue_name}: weight={weight:.4f}")
else:
    print("No attention breakdown available")

## 5. Summary Statistics

In [ ]:
# Summary statistics
summary_stats = pd.DataFrame({
    'Metric': [
        'Average Max Weight',
        'Average Entropy',
        'Routing Accuracy',
        'Number of Active Tissues (>5% weight)',
        'Temperature'
    ],
    'Value': [
        f"{max_weights.mean():.4f}",
        f"{entropy.mean():.4f}",
        f"{accuracy:.2%}" if len(true_tissues) > 0 else 'N/A',
        f"{(tissue_weights > 0.05).sum(axis=1).mean():.2f}",
        f"{model.tissue_router.temperature.item():.4f}"
    ]
})

print("\nSummary Statistics:")
print(summary_stats.to_string(index=False))

# Save to file
summary_stats.to_csv('../output/tissue_routing_summary.csv', index=False)